# Rural primary-care workforce and infant mortality

This notebook adds a same-reference-year state comparison using RBI Table 16 (2023 rural PHC doctor staffing) and RBI Table 4 (2023 infant mortality rate). It measures staffing against the source-reported requirement, not against population. The analysis is cross-sectional and observational; it is not a causal estimate.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

DATA = Path('../Dataset')
staff = pd.read_csv(DATA / 'state_rural_phc_doctors_2023.csv')
imr = pd.read_csv(DATA / 'state_infant_mortality_rate_2023.csv')
staff.head()

## 1. Validate and harmonise state labels

The RBI workforce table combines Dadra & Nagar Haveli and Daman & Diu; the IMR table labels the same combined observation as Dadra & Nagar Haveli (includes Daman & Diu). Missing source markers remain missing, and the all-India aggregate is excluded.

In [ ]:
imr = imr.loc[~imr['state_ut'].eq('All India')].copy()
imr = imr.dropna(subset=['infant_mortality_rate_2023_per_1000_live_births'])
staff['state_key'] = staff['state_ut'].replace({'Dadra & Nagar Haveli & Daman & Diu': 'Dadra & Nagar Haveli (includes Daman & Diu)'})
imr['state_key'] = imr['state_ut']
df = staff.merge(imr[['state_key','infant_mortality_rate_2023_per_1000_live_births']], on='state_key', how='inner', validate='one_to_one')
df['phc_doctor_coverage_ratio'] = df['in_position_rural_phc_doctors'] / df['required_rural_phc_doctors'].where(df['required_rural_phc_doctors'] > 0)
print('Matched rows:', len(df))
print('Rows with calculable staffing ratio:', df['phc_doctor_coverage_ratio'].notna().sum())
print('Unmatched workforce rows:', sorted(set(staff.state_key) - set(df.state_key)))
assert df['state_key'].is_unique
df[['state_ut','required_rural_phc_doctors','in_position_rural_phc_doctors','phc_doctor_coverage_ratio','infant_mortality_rate_2023_per_1000_live_births']].head()

## 2. Staffing adequacy relative to reported requirement

A ratio of 1.0 means the number of doctors in position equals the reported requirement; below 1.0 indicates fewer doctors in position than required; above 1.0 indicates staffing above the requirement. This does not measure population coverage or actual access.

In [ ]:
valid = df.dropna(subset=['phc_doctor_coverage_ratio']).copy()
valid = valid.sort_values('phc_doctor_coverage_ratio')
display(valid[['state_ut','required_rural_phc_doctors','in_position_rural_phc_doctors','phc_doctor_coverage_ratio']].round(3))
print('Median staffing coverage ratio:', round(valid.phc_doctor_coverage_ratio.median(), 3))
print('States/UTs below reported requirement:', int((valid.phc_doctor_coverage_ratio < 1).sum()), 'of', len(valid))
print('States/UTs at or above reported requirement:', int((valid.phc_doctor_coverage_ratio >= 1).sum()), 'of', len(valid))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(valid.state_ut, valid.phc_doctor_coverage_ratio)
ax.axvline(1, linestyle='--', linewidth=1, color='black', label='Reported requirement met')
ax.set_xlabel('Doctors in position / required doctors')
ax.set_title('Rural PHC doctor staffing relative to reported requirement, 2023')
ax.legend()
fig.tight_layout()
plt.show()

## 3. Staffing coverage and infant mortality

The two indicators refer to 2023. Pearson correlation measures linear association; Spearman correlation measures rank association and is less sensitive to the exact scale. Neither controls for socioeconomic determinants or establishes causality.

In [ ]:
analysis = df.dropna(subset=['phc_doctor_coverage_ratio']).copy()
x = analysis['phc_doctor_coverage_ratio']
y = analysis['infant_mortality_rate_2023_per_1000_live_births']
pearson_r, pearson_p = pearsonr(x, y)
spearman_rho, spearman_p = spearmanr(x, y)
print(f'Pearson r = {pearson_r:.3f}; p-value = {pearson_p:.4f}; n = {len(analysis)}')
print(f'Spearman rho = {spearman_rho:.3f}; p-value = {spearman_p:.4f}; n = {len(analysis)}')
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(x, y, alpha=0.8)
for _, row in analysis.iterrows():
    ax.annotate(row['state_ut'], (row['phc_doctor_coverage_ratio'], row['infant_mortality_rate_2023_per_1000_live_births']), fontsize=7, xytext=(3,3), textcoords='offset points')
ax.set_xlabel('Rural PHC doctor staffing coverage ratio, 2023')
ax.set_ylabel('Infant mortality rate per 1,000 live births, 2023')
ax.set_title('Rural PHC doctor staffing and infant mortality across states/UTs')
fig.tight_layout()
plt.show()

## 4. Interpretation and limitations

Interpret the coefficient only after checking the plotted observations and the matched sample. A negative association would mean states with higher PHC doctor staffing relative to the stated requirement tend to have lower IMR in this sample. It would not establish a causal effect.

Important limitations:
- Rural PHC doctors are only one component of the health system; this excludes private providers, urban facilities, nurses, specialists outside PHCs and service quality.
- The staffing ratio uses the administrative requirement as its denominator. It is not a population-adjusted doctor density.
- IMR is affected by income, maternal education, nutrition, sanitation, fertility, transport and other determinants.
- The analysis is cross-sectional, and small states/UTs may have volatile rates.
- Missing asterisk/dash values in the source remain missing; they are not imputed as zero.

A stronger next step would combine several years of state-level workforce and outcome data and add socioeconomic controls, after verifying consistent definitions and coverage.

In [ ]:
# Reproducibility checks
assert df.state_key.is_unique
assert len(analysis) >= 25
assert analysis['phc_doctor_coverage_ratio'].notna().all()
assert analysis['infant_mortality_rate_2023_per_1000_live_births'].notna().all()
print('Notebook checks passed.')